# LSTM Model - WTI Crude Oil Next-Day Return Prediction

This notebook builds an LSTM model to forecast next-day WTI crude oil log returns using the full feature set (70 features). It uses:
- **Walk-forward validation** with train/val/test split by date
- **Grid search** with TimeSeriesSplit for hyperparameter tuning
- **Evaluation metrics:** RMSE, MAE, and directional accuracy

## 1. Imports & Setup

In [38]:
import pandas as pd
import numpy as np
import json
import warnings
from pathlib import Path
from itertools import product
from copy import deepcopy

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Paths
NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent.parent.parent if NOTEBOOK_DIR.name == 'return_prediction' else NOTEBOOK_DIR
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
RESULTS_DIR = PROJECT_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

print(f'Project dir: {PROJECT_DIR}')
print(f'Data dir:    {PROCESSED_DIR}')

Using device: cpu
Project dir: c:\Users\Account\Desktop\Leone\MSc CF\Data Science\Group\data_science_crude_oil
Data dir:    c:\Users\Account\Desktop\Leone\MSc CF\Data Science\Group\data_science_crude_oil\data\processed


## 2. Load & Preprocess Data

In [ ]:
# ── DATASET PICKER ────────────────────────────────────────────────────
# Switch dataset here. Comment/uncomment ONE line:
# DATASET = 'sanity_data'          # full: 76 features (28 Fed raw + 5D changes)
# DATASET = 'sanity_data_core'     # core: 13 features (returns, lags, vol)
DATASET = 'sanity_data_pca'        # pca:  51 features (Fed text compressed to 3 PCA components)
# ──────────────────────────────────

df = pd.read_parquet(PROCESSED_DIR / f'{DATASET}.parquet')
df['Date'] = pd.to_datetime(df['Date'])

TARGET = 'Target'
NON_FEATURE_COLS = ['Date', 'Target', 'WTI_Close', 'Brent_Close']
FEATURE_COLS = [c for c in df.columns if c not in NON_FEATURE_COLS]
df = df.dropna(subset=[TARGET] + FEATURE_COLS).reset_index(drop=True)

print(f'Dataset: {DATASET}')
print(f'Shape:   {df.shape}')
print(f'Features: {len(FEATURE_COLS)}')
print(f'Target:  {TARGET} (next-day WTI log return)')
print(f'Date range: {df["Date"].min().date()} → {df["Date"].max().date()}')

## 3. Train / Val / Test Split

In [ ]:
TRAIN_END = '2022-12-31'
VAL_END = '2024-06-30'

train_df = df[df['Date'] <= TRAIN_END].copy()
val_df = df[(df['Date'] > TRAIN_END) & (df['Date'] <= VAL_END)].copy()
test_df = df[df['Date'] > VAL_END].copy()

print(f'Train: {len(train_df)} rows  ({train_df["Date"].min().date()} to {train_df["Date"].max().date()})')
print(f'Val:   {len(val_df)} rows  ({val_df["Date"].min().date()} to {val_df["Date"].max().date()})')
print(f'Test:  {len(test_df)} rows  ({test_df["Date"].min().date()} to {test_df["Date"].max().date()})')
print(f'\nTrain: {len(train_df)/len(df)*100:.1f}% | Val: {len(val_df)/len(df)*100:.1f}% | Test: {len(test_df)/len(df)*100:.1f}%')

## 4. Feature Scaling

In [ ]:
# Fit scalers on training data only
scaler_X = RobustScaler()
scaler_y = RobustScaler()

train_X = scaler_X.fit_transform(train_df[FEATURE_COLS].values)
train_y = scaler_y.fit_transform(train_df[[TARGET]].values).flatten()

val_X = scaler_X.transform(val_df[FEATURE_COLS].values)
val_y = scaler_y.transform(val_df[[TARGET]].values).flatten()

test_X = scaler_X.transform(test_df[FEATURE_COLS].values)
test_y = scaler_y.transform(test_df[[TARGET]].values).flatten()

print(f'Scaled train features shape: {train_X.shape}')
print(f'Scaled val features shape:   {val_X.shape}')
print(f'Scaled test features shape:  {test_X.shape}')

## 5. Sequence Creation

In [42]:
def create_sequences(features, target, seq_len):
    """
    Create LSTM input sequences.
    X[i] = features[i : i+seq_len]  (shape: seq_len x n_features)
    y[i] = target[i + seq_len - 1]  (the target at the last timestep)
    
    Note: Target_Next_Day_Return is already the next day's return,
    so we use the target at the last timestep of each window.
    """
    X, y = [], []
    for i in range(len(features) - seq_len + 1):
        X.append(features[i:i + seq_len])
        y.append(target[i + seq_len - 1])
    return np.array(X), np.array(y)


def create_sequences_with_bridge(prev_features, prev_target, curr_features, curr_target, seq_len):
    """
    Create sequences for val/test sets, bridging from the prior split
    so we don't lose the first (seq_len-1) sequences.
    
    The bridge prepends (seq_len-1) rows from the previous split,
    giving the first sequences in the current split a full lookback
    window. All resulting targets fall within the current split.
    """
    # Prepend tail of previous split
    bridge_X = np.vstack([prev_features[-(seq_len - 1):], curr_features])
    bridge_y = np.hstack([prev_target[-(seq_len - 1):], curr_target])
    
    X_all, y_all = create_sequences(bridge_X, bridge_y, seq_len)
    
    # All targets already fall in the current split (bridge_y[seq_len-1:]
    # are all from curr_target), so no further slicing needed.
    return X_all, y_all


# Quick test
X_test_seq, y_test_seq = create_sequences(train_X, train_y, seq_len=10)
print(f'Example: seq_len=10, train sequences: {X_test_seq.shape}, targets: {y_test_seq.shape}')

# Verify bridge: should produce exactly len(val_y) sequences
X_bridge_test, y_bridge_test = create_sequences_with_bridge(train_X, train_y, val_X, val_y, seq_len=10)
print(f'Bridge test: val rows={len(val_y)}, bridge sequences={X_bridge_test.shape[0]} (should match)')

Example: seq_len=10, train sequences: (1226, 10, 70), targets: (1226,)
Bridge test: val rows=312, bridge sequences=312 (should match)


## 6. PyTorch Dataset & Model

In [ ]:
class OilDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y).unsqueeze(1)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


ACTIVATIONS = {
    'relu': nn.ReLU(),
    'tanh': nn.Tanh(),
    'none': nn.Identity(),
}


class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout, activation='none', fc_hidden=0):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.activation = ACTIVATIONS[activation]
        
        if fc_hidden > 0:
            self.head = nn.Sequential(
                nn.Linear(hidden_size, fc_hidden),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(fc_hidden, 1),
            )
        else:
            self.head = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        lstm_out, (h_n, _) = self.lstm(x)
        out = self.dropout(h_n[-1])  # last layer's final hidden state
        out = self.activation(out)
        return self.head(out)

## 7. Training & Evaluation Functions

In [44]:
def train_model(model, train_loader, val_loader, epochs, lr, patience=10, verbose=False):
    """Train LSTM with early stopping on validation loss."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    best_val_loss = float('inf')
    best_state = None
    wait = 0
    train_losses, val_losses = [], []
    
    for epoch in range(epochs):
        # Training
        model.train()
        epoch_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            pred = model(X_batch)
            loss = criterion(pred, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                val_loss += criterion(model(X_b), y_b).item()
        
        avg_train = epoch_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)
        train_losses.append(avg_train)
        val_losses.append(avg_val)
        
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state = deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                if verbose:
                    print(f'  Early stopping at epoch {epoch+1}')
                break
        
        if verbose and (epoch + 1) % 20 == 0:
            print(f'  Epoch {epoch+1}: train={avg_train:.6f}, val={avg_val:.6f}')
    
    model.load_state_dict(best_state)
    return model, train_losses, val_losses


def evaluate(model, loader, scaler_y):
    """Compute RMSE, MAE, directional accuracy on inverse-scaled predictions."""
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            X_b = X_b.to(device)
            p = model(X_b).cpu().numpy()
            preds.append(p)
            actuals.append(y_b.numpy())
    
    preds = np.concatenate(preds).flatten()
    actuals = np.concatenate(actuals).flatten()
    
    # Inverse transform to original scale
    preds_inv = scaler_y.inverse_transform(preds.reshape(-1, 1)).flatten()
    actuals_inv = scaler_y.inverse_transform(actuals.reshape(-1, 1)).flatten()
    
    rmse = np.sqrt(mean_squared_error(actuals_inv, preds_inv))
    mae = mean_absolute_error(actuals_inv, preds_inv)
    
    # Directional accuracy
    dir_acc = np.mean(np.sign(preds_inv) == np.sign(actuals_inv))
    
    return {
        'RMSE': rmse,
        'MAE': mae,
        'Dir_Acc': dir_acc,
        'preds': preds_inv,
        'actuals': actuals_inv
    }

In [ ]:
def run_synthetic_test(name, features, targets, seq_len=10, hidden=32, epochs=100, lr=1e-3):
    """Train LSTM on synthetic data and report RMSE."""
    n = len(features)
    split = int(n * 0.8)
    
    # Create sequences
    X_seq, y_seq = [], []
    for i in range(len(features) - seq_len):
        X_seq.append(features[i:i+seq_len])
        y_seq.append(targets[i+seq_len-1])
    X_seq, y_seq = np.array(X_seq), np.array(y_seq)
    
    X_tr, y_tr = X_seq[:split], y_seq[:split]
    X_te, y_te = X_seq[split:], y_seq[split:]
    
    tr_loader = DataLoader(OilDataset(X_tr, y_tr), batch_size=32, shuffle=False)
    te_loader = DataLoader(OilDataset(X_te, y_te), batch_size=32, shuffle=False)
    
    n_feat = features.shape[1]
    torch.manual_seed(SEED)
    model = LSTMModel(n_feat, hidden, 1, 0.0, activation='none').to(device)
    
    # Dummy scaler (identity) for evaluate()
    class IdentityScaler:
        def inverse_transform(self, x): return x
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    for epoch in range(epochs):
        model.train()
        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            optimizer.step()
    
    results = evaluate(model, te_loader, IdentityScaler())
    target_std = y_te.std()
    ratio = results['RMSE'] / target_std
    status = 'PASS' if ratio < 0.3 else 'MARGINAL' if ratio < 0.6 else 'FAIL'
    
    print(f"  {name:25s}  RMSE={results['RMSE']:.6f}  target_std={target_std:.6f}  "
          f"RMSE/std={ratio:.4f}  [{status}]")
    return results

print("=" * 80)
print("SYNTHETIC DIAGNOSTIC: Testing LSTM on known-signal data")
print("=" * 80)

np.random.seed(SEED)
N = 2000

# --- Test 1: Sine wave ---
# y[t] = sin(2*pi*t/50), features = [y[t]] (feed the series itself)
t = np.arange(N, dtype=float)
sine = np.sin(2 * np.pi * t / 50)
sine_features = sine.reshape(-1, 1)
sine_targets = sine  # predict the current value from the lookback window

run_synthetic_test("Sine wave", sine_features, sine_targets)

# --- Test 2: AR(2) process ---
# y[t] = 0.5*y[t-1] - 0.3*y[t-2] + noise
ar = np.zeros(N)
for i in range(2, N):
    ar[i] = 0.5 * ar[i-1] - 0.3 * ar[i-2] + np.random.randn() * 0.1
ar_features = ar.reshape(-1, 1)
ar_targets = ar

run_synthetic_test("AR(2) process", ar_features, ar_targets)

# --- Test 3: Identity (target = feature) ---
# Feed the target itself as a feature -- LSTM should get near-zero RMSE
signal = np.random.randn(N) * 0.03  # similar scale to oil returns
identity_features = np.column_stack([signal, np.random.randn(N, 3) * 0.03])  # target + noise features
identity_targets = signal

run_synthetic_test("Identity (target=feature)", identity_features, identity_targets)

# --- Test 4: Linear combination ---
# y = 0.7*x1 + 0.3*x2 + noise
x1 = np.random.randn(N) * 0.03
x2 = np.random.randn(N) * 0.03
y_linear = 0.7 * x1 + 0.3 * x2 + np.random.randn(N) * 0.005
linear_features = np.column_stack([x1, x2])

run_synthetic_test("Linear combo (0.7*x1+0.3*x2)", linear_features, y_linear)

print("\nIf all tests PASS, the LSTM code is correct.")
print("Poor performance on real data means the signal is too weak, not a code bug.")

## 7b. Synthetic Data Diagnostic

Three sanity tests to confirm the LSTM code works correctly:

1. **Sine wave** -- can it learn a deterministic temporal pattern?
2. **AR(2) process** -- can it learn a known linear recurrence?
3. **Identity** -- if you feed the target as a feature, does it get near-perfect RMSE?

If all three pass, the LSTM is working and the real data is just hard to predict.

## 8. Hyperparameter Grid Search

Grid search using `TimeSeriesSplit(n_splits=3)` within the training set.

In [ ]:
param_grid = {
    'seq_len': [3, 5, 7, 10, 15, 20, 30, 40, 60],
    'hidden_size': [16, 32, 64, 96, 128],
    'num_layers': [1, 2, 3],
    'dropout': [0.0, 0.1, 0.3, 0.5],
    'learning_rate': [1e-3, 5e-4],
    'activation': ['tanh', 'relu'],
    'fc_hidden': [0, 32],
}

EPOCHS = 50
BATCH_SIZE = 32

keys = list(param_grid.keys())
values = list(param_grid.values())
all_combos = list(product(*values))
print(f'Total configurations: {len(all_combos)}')
print(f'Total training runs:  {len(all_combos) * 3} (3 folds each)')

In [ ]:
import time as _time
results = []
_t0 = _time.time()
_best_so_far = float('inf')

for i, combo in enumerate(all_combos):
    params = dict(zip(keys, combo))
    sl = params['seq_len']
    
    # Create sequences from full training data
    X_all, y_all = create_sequences(train_X, train_y, sl)
    
    # 3-fold TimeSeriesSplit
    tscv = TimeSeriesSplit(n_splits=3)
    fold_val_losses = []
    
    for fold_train_idx, fold_val_idx in tscv.split(X_all):
        X_tr, y_tr = X_all[fold_train_idx], y_all[fold_train_idx]
        X_vl, y_vl = X_all[fold_val_idx], y_all[fold_val_idx]
        
        tr_loader = DataLoader(OilDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=False)
        vl_loader = DataLoader(OilDataset(X_vl, y_vl), batch_size=BATCH_SIZE, shuffle=False)
        
        torch.manual_seed(SEED)
        model = LSTMModel(
            input_size=len(FEATURE_COLS),
            hidden_size=params['hidden_size'],
            num_layers=params['num_layers'],
            dropout=params['dropout'],
            activation=params['activation'],
            fc_hidden=params['fc_hidden']
        ).to(device)
        
        model, _, v_losses = train_model(
            model, tr_loader, vl_loader,
            epochs=EPOCHS, lr=params['learning_rate'], patience=10
        )
        fold_val_losses.append(min(v_losses))
    
    avg_cv_loss = np.mean(fold_val_losses)
    results.append({**params, 'avg_cv_val_loss': avg_cv_loss})
    
    _is_best = avg_cv_loss < _best_so_far
    if _is_best:
        _best_so_far = avg_cv_loss
    
    if (i + 1) % 25 == 0 or (i + 1) == len(all_combos) or _is_best:
        _elapsed = _time.time() - _t0
        _rate = (i + 1) / _elapsed
        _eta = (len(all_combos) - i - 1) / _rate if _rate > 0 else 0
        _marker = ' *** NEW BEST ***' if _is_best else ''
        print(f'[{i+1:5d}/{len(all_combos)}] loss={avg_cv_loss:.6f} | '
              f'{_elapsed/60:.1f}m elapsed, ETA {_eta/60:.1f}m ({_eta/3600:.1f}h){_marker}')

print(f'\nLSTM grid search complete. {len(results)} configs in {(_time.time()-_t0)/60:.1f} minutes.')

In [47]:
# Results summary
results_df = pd.DataFrame(results).sort_values('avg_cv_val_loss').reset_index(drop=True)
print('Top 10 configurations:\n')
print(results_df.head(10).to_string(index=False))

Top 10 configurations:

 seq_len  hidden_size  num_layers  dropout  learning_rate  avg_cv_val_loss
      20           64           2      0.2         0.0010       517.062753
      10           64           2      0.2         0.0010       517.073415
      20           64           2      0.3         0.0010       517.467367
       5           64           2      0.2         0.0010       517.658767
      10           64           2      0.3         0.0005       517.935953
       5           64           2      0.3         0.0010       517.935967
      10           64           2      0.3         0.0010       517.987452
      20           64           2      0.3         0.0005       518.006503
      20           64           2      0.2         0.0005       518.242814
       5           64           2      0.2         0.0005       518.308136


## 9. Train Final Model with Best Hyperparameters

In [ ]:
# Extract best hyperparameters
best = results_df.iloc[0]
best_sl = int(best['seq_len'])
best_hidden = int(best['hidden_size'])
best_layers = int(best['num_layers'])
best_dropout = float(best['dropout'])
best_lr = float(best['learning_rate'])
best_act = str(best['activation'])
best_fc = int(best['fc_hidden'])

print('Best hyperparameters:')
print(f'  seq_len:       {best_sl}')
print(f'  hidden_size:   {best_hidden}')
print(f'  num_layers:    {best_layers}')
print(f'  dropout:       {best_dropout}')
print(f'  learning_rate: {best_lr}')
print(f'  activation:    {best_act}')
print(f'  fc_hidden:     {best_fc}')
print(f'  CV val loss:   {best["avg_cv_val_loss"]:.6f}')

In [ ]:
# Create sequences
X_train_seq, y_train_seq = create_sequences(train_X, train_y, best_sl)
X_val_seq, y_val_seq = create_sequences_with_bridge(train_X, train_y, val_X, val_y, best_sl)

print(f'Train sequences: {X_train_seq.shape}')
print(f'Val sequences:   {X_val_seq.shape}')

train_loader = DataLoader(OilDataset(X_train_seq, y_train_seq), batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(OilDataset(X_val_seq, y_val_seq), batch_size=BATCH_SIZE, shuffle=False)

# Train final model
torch.manual_seed(SEED)
final_model = LSTMModel(
    input_size=len(FEATURE_COLS),
    hidden_size=best_hidden,
    num_layers=best_layers,
    dropout=best_dropout,
    activation=best_act,
    fc_hidden=best_fc
).to(device)

final_model, train_losses, val_losses = train_model(
    final_model, train_loader, val_loader,
    epochs=EPOCHS, lr=best_lr, patience=15, verbose=True
)

# Validation metrics
val_metrics = evaluate(final_model, val_loader, scaler_y)
print(f'\nValidation Results:')
print(f'  RMSE: {val_metrics["RMSE"]:.6f}')
print(f'  MAE:  {val_metrics["MAE"]:.6f}')
print(f'  Directional Accuracy: {val_metrics["Dir_Acc"]:.4f}')

## 10. Training Curves

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(train_losses, label='Train Loss', linewidth=1.5)
ax.plot(val_losses, label='Val Loss', linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('LSTM Training & Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'lstm_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Test Set Evaluation

In [ ]:
# Create test sequences (bridge from val set)
X_test_seq, y_test_seq = create_sequences_with_bridge(val_X, val_y, test_X, test_y, best_sl)
print(f'Test sequences: {X_test_seq.shape}')

test_loader = DataLoader(OilDataset(X_test_seq, y_test_seq), batch_size=BATCH_SIZE, shuffle=False)

# Test metrics
test_metrics = evaluate(final_model, test_loader, scaler_y)
print(f'\nTest Results:')
print(f'  RMSE: {test_metrics["RMSE"]:.6f}')
print(f'  MAE:  {test_metrics["MAE"]:.6f}')
print(f'  Directional Accuracy: {test_metrics["Dir_Acc"]:.4f}')

## XGBoost -- Grid Search

In [ ]:
xgb_param_grid = {
    'n_estimators':     [50, 100, 150, 200, 300, 500],
    'max_depth':        [2, 3, 4, 5, 6, 7],
    'learning_rate':    [0.01, 0.05, 0.1, 0.2],
    'subsample':        [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8],
    'reg_alpha':        [0, 0.1],
    'reg_lambda':       [0.5, 1.0, 2.0],
}

xgb_keys = list(xgb_param_grid.keys())
xgb_values = list(xgb_param_grid.values())
xgb_combos = list(product(*xgb_values))
print(f'XGBoost grid search: {len(xgb_combos)} configurations')

In [ ]:
xgb_results = []
_t0_xgb = _time.time()
_best_xgb_so_far = float('inf')

try:
    for i, combo in enumerate(xgb_combos):
        params = dict(zip(xgb_keys, combo))
        tscv = TimeSeriesSplit(n_splits=3)
        fold_losses = []

        for tr_idx, vl_idx in tscv.split(train_X):
            xgb = XGBRegressor(
                **params,
                objective='reg:squarederror',
                random_state=SEED,
                verbosity=0,
            )
            xgb.fit(
                train_X[tr_idx], train_y[tr_idx],
                eval_set=[(train_X[vl_idx], train_y[vl_idx])],
                verbose=False,
            )
            preds = xgb.predict(train_X[vl_idx])
            fold_losses.append(mean_squared_error(train_y[vl_idx], preds))

        avg_loss = np.mean(fold_losses)
        xgb_results.append({**params, 'avg_cv_val_loss': avg_loss})

        _is_best = avg_loss < _best_xgb_so_far
        if _is_best:
            _best_xgb_so_far = avg_loss

        if (i + 1) % 100 == 0 or (i + 1) == len(xgb_combos) or _is_best:
            _elapsed = _time.time() - _t0_xgb
            _rate = (i + 1) / _elapsed
            _eta = (len(xgb_combos) - i - 1) / _rate if _rate > 0 else 0
            _marker = ' *** NEW BEST ***' if _is_best else ''
            print(f'[{i+1:5d}/{len(xgb_combos)}] loss={avg_loss:.6f} | '
                  f'{_elapsed/60:.1f}m elapsed, ETA {_eta/60:.1f}m ({_eta/3600:.1f}h){_marker}')

except KeyboardInterrupt:
    print(f'\n*** Interrupted after {len(xgb_results)}/{len(xgb_combos)} configs ***')

finally:
    if xgb_results:
        xgb_results_df = pd.DataFrame(xgb_results).sort_values('avg_cv_val_loss').reset_index(drop=True)
        xgb_results_df.to_csv(RESULTS_DIR / 'xgb_grid_search_results.csv', index=False)

print(f'\nXGBoost grid search complete. {len(xgb_results)} configs in {(_time.time()-_t0_xgb)/60:.1f} minutes.')
print(f'\nTop 5 XGBoost configurations:')
print(xgb_results_df.head().to_string(index=False))

In [ ]:
best_xgb = xgb_results_df.iloc[0].drop('avg_cv_val_loss').to_dict()
best_xgb['n_estimators'] = int(best_xgb['n_estimators'])
best_xgb['max_depth'] = int(best_xgb['max_depth'])

print('Best XGBoost hyperparameters:')
for k, v in best_xgb.items():
    print(f'  {k:<20} {v}')

final_xgb = XGBRegressor(
    **best_xgb,
    objective='reg:squarederror',
    random_state=SEED,
    verbosity=0,
    early_stopping_rounds=20,
)
final_xgb.fit(
    train_X, train_y,
    eval_set=[(val_X, val_y)],
    verbose=False,
)
print(f'\nBest iteration: {final_xgb.best_iteration}')

In [ ]:
# XGBoost predictions on full test set
xgb_all_preds_scaled = final_xgb.predict(test_X)

# Inverse transform to original scale
xgb_all_preds = scaler_y.inverse_transform(xgb_all_preds_scaled.reshape(-1, 1)).flatten()
xgb_all_actuals = scaler_y.inverse_transform(test_y.reshape(-1, 1)).flatten()

# Trim to match LSTM's test range (LSTM loses seq_len-1 obs from front)
lstm_n_test = len(test_metrics['actuals'])
xgb_test_preds = xgb_all_preds[-lstm_n_test:]
xgb_test_actuals = xgb_all_actuals[-lstm_n_test:]

xgb_rmse = np.sqrt(mean_squared_error(xgb_test_actuals, xgb_test_preds))
xgb_mae = mean_absolute_error(xgb_test_actuals, xgb_test_preds)
xgb_dir = np.mean(np.sign(xgb_test_preds) == np.sign(xgb_test_actuals))

print(f'XGBoost Test Results (aligned with LSTM, {lstm_n_test} obs):')
print(f'  RMSE: {xgb_rmse:.6f}')
print(f'  MAE:  {xgb_mae:.6f}')
print(f'  Directional Accuracy: {xgb_dir:.4f}')

In [ ]:
fi = final_xgb.feature_importances_
top_n = min(20, len(FEATURE_COLS))
top_idx = np.argsort(fi)[-top_n:]

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(
    [FEATURE_COLS[i] for i in top_idx],
    fi[top_idx],
    color='#375623', alpha=0.8
)
ax.set_xlabel('Feature importance (gain)')
ax.set_title(f'XGBoost Top {top_n} Feature Importances', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'xgb_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nTop {top_n} features by importance:')
for idx in reversed(top_idx):
    print(f'  {FEATURE_COLS[idx]:<35} {fi[idx]:.4f}')

In [ ]:
# AR(10): predict target from 10 lagged returns using OLS
ar_order = 10
target_series = df[TARGET].values

# Construct lagged feature matrix
ar_X = np.column_stack([np.roll(target_series, i) for i in range(1, ar_order + 1)])
ar_y = target_series.copy()

# Mask out the first 10 rows (incomplete lags)
valid = np.arange(ar_order, len(target_series))
ar_X = ar_X[valid]
ar_y = ar_y[valid]
ar_dates = df['Date'].values[valid]

# Split using the same date boundaries as the LSTM
train_mask = ar_dates <= np.datetime64(TRAIN_END)
test_mask = ar_dates > np.datetime64(VAL_END)

ar_X_train, ar_y_train = ar_X[train_mask], ar_y[train_mask]
ar_X_test, ar_y_test = ar_X[test_mask], ar_y[test_mask]
ar_test_dates = ar_dates[test_mask]

# Fit OLS: y = X @ beta + intercept
ar_X_train_i = np.column_stack([np.ones(len(ar_X_train)), ar_X_train])
ar_X_test_i = np.column_stack([np.ones(len(ar_X_test)), ar_X_test])

beta = np.linalg.lstsq(ar_X_train_i, ar_y_train, rcond=None)[0]

print(f'AR(10) coefficients:')
print(f'  Intercept: {beta[0]:.6f}')
for i in range(ar_order):
    print(f'  Lag {i+1:2d}:    {beta[i+1]:+.6f}')

# Predict on test set
ar_preds = ar_X_test_i @ beta

# Metrics
ar_rmse = np.sqrt(mean_squared_error(ar_y_test, ar_preds))
ar_mae = mean_absolute_error(ar_y_test, ar_preds)
ar_dir = np.mean(np.sign(ar_preds) == np.sign(ar_y_test))

print(f'\nAR(10) Test Results:')
print(f'  RMSE: {ar_rmse:.6f}')
print(f'  MAE:  {ar_mae:.6f}')
print(f'  Directional Accuracy: {ar_dir:.4f}')
print(f'  Test observations: {len(ar_y_test)}')

## 12. Baseline Comparisons

In [ ]:
test_actuals = test_metrics['actuals']
test_preds = test_metrics['preds']

# Naive baseline: predict 0 (no change)
naive_rmse = np.sqrt(mean_squared_error(test_actuals, np.zeros_like(test_actuals)))
naive_mae = mean_absolute_error(test_actuals, np.zeros_like(test_actuals))
naive_dir = 0.5  # random guess benchmark

# Persistence baseline: predict yesterday's return
persist_preds = test_actuals[:-1]
persist_actuals = test_actuals[1:]
persist_rmse = np.sqrt(mean_squared_error(persist_actuals, persist_preds))
persist_mae = mean_absolute_error(persist_actuals, persist_preds)
persist_dir = np.mean(np.sign(persist_preds) == np.sign(persist_actuals))

# LSTM on same N-1 subset for fair persistence comparison
lstm_preds_sub = test_preds[1:]
lstm_actuals_sub = test_actuals[1:]
lstm_dir_sub = np.mean(np.sign(lstm_preds_sub) == np.sign(lstm_actuals_sub))

print('Model Comparison on Test Set:')
print(f'{"":15s} {"RMSE":<12} {"MAE":<12} {"Dir Acc":<12}')
print("-" * 50)
print(f'{"LSTM":<15} {test_metrics["RMSE"]:<12.6f} {test_metrics["MAE"]:<12.6f} {test_metrics["Dir_Acc"]:<12.4f}')
print(f'{"XGBoost":<15} {xgb_rmse:<12.6f} {xgb_mae:<12.6f} {xgb_dir:<12.4f}')
print(f'{"AR(10)":<15} {ar_rmse:<12.6f} {ar_mae:<12.6f} {ar_dir:<12.4f}')
print(f'{"Naive (0)":<15} {naive_rmse:<12.6f} {naive_mae:<12.6f} {naive_dir:<12.4f}')
print(f'{"Persistence":<15} {persist_rmse:<12.6f} {persist_mae:<12.6f} {persist_dir:<12.4f}')
print(f'\nNote: Persistence uses N-1 obs. LSTM Dir Acc on same subset: {lstm_dir_sub:.4f}')

## 13. Visualizations

In [ ]:
# Predictions vs Actuals - Time Series (LSTM + XGBoost + AR(10))
test_dates = test_df['Date'].values[-len(test_actuals):]

fig, axes = plt.subplots(4, 1, figsize=(14, 16))

# --- Panel 1: Time series overlay ---
axes[0].plot(test_dates, test_actuals, label='Actual', alpha=0.7, linewidth=1, color='#1f4e79')
axes[0].plot(test_dates, test_preds, label='LSTM', alpha=0.7, linewidth=1, color='#c55a11')
axes[0].plot(test_dates, xgb_test_preds, label='XGBoost', alpha=0.7, linewidth=1, color='#8e44ad')
axes[0].plot(ar_test_dates, ar_preds, label='AR(10)', alpha=0.7, linewidth=1, color='#375623')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Log Return')
axes[0].set_title('Actual vs LSTM vs XGBoost vs AR(10) - Test Set')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- Panel 2: Scatter LSTM ---
axes[1].scatter(test_actuals, test_preds, alpha=0.5, s=20, color='#c55a11', label='LSTM')
lims = [min(test_actuals.min(), test_preds.min()), max(test_actuals.max(), test_preds.max())]
axes[1].plot(lims, lims, 'r--', linewidth=1, label='Perfect prediction')
axes[1].set_xlabel('Actual Log Return')
axes[1].set_ylabel('Predicted Log Return')
axes[1].set_title('LSTM: Predicted vs Actual')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# --- Panel 3: Scatter XGBoost ---
axes[2].scatter(xgb_test_actuals, xgb_test_preds, alpha=0.5, s=20, color='#8e44ad', label='XGBoost')
lims_xgb = [min(xgb_test_actuals.min(), xgb_test_preds.min()), max(xgb_test_actuals.max(), xgb_test_preds.max())]
axes[2].plot(lims_xgb, lims_xgb, 'r--', linewidth=1, label='Perfect prediction')
axes[2].set_xlabel('Actual Log Return')
axes[2].set_ylabel('Predicted Log Return')
axes[2].set_title('XGBoost: Predicted vs Actual')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

# --- Panel 4: Scatter AR(10) ---
axes[3].scatter(ar_y_test, ar_preds, alpha=0.5, s=20, color='#375623', label='AR(10)')
lims2 = [min(ar_y_test.min(), ar_preds.min()), max(ar_y_test.max(), ar_preds.max())]
axes[3].plot(lims2, lims2, 'r--', linewidth=1, label='Perfect prediction')
axes[3].set_xlabel('Actual Log Return')
axes[3].set_ylabel('Predicted Log Return')
axes[3].set_title('AR(10): Predicted vs Actual')
axes[3].legend()
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'lstm_predictions_vs_actuals.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Rolling directional accuracy for all models ---
correct_lstm = (np.sign(test_preds) == np.sign(test_actuals)).astype(float)
rolling_lstm = pd.Series(correct_lstm).rolling(window=20, min_periods=10).mean()

correct_xgb = (np.sign(xgb_test_preds) == np.sign(xgb_test_actuals)).astype(float)
rolling_xgb = pd.Series(correct_xgb).rolling(window=20, min_periods=10).mean()

correct_ar = (np.sign(ar_preds) == np.sign(ar_y_test)).astype(float)
rolling_ar = pd.Series(correct_ar).rolling(window=20, min_periods=10).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_dates, rolling_lstm.values, linewidth=1.5, color='#c55a11', label='LSTM')
ax.plot(test_dates, rolling_xgb.values, linewidth=1.5, color='#8e44ad', label='XGBoost')
ax.plot(ar_test_dates, rolling_ar.values, linewidth=1.5, color='#375623', label='AR(10)')
ax.axhline(y=0.5, color='red', linestyle='--', linewidth=1, label='Random (50%)')
ax.set_xlabel('Date')
ax.set_ylabel('Directional Accuracy')
ax.set_title('Rolling 20-Day Directional Accuracy - LSTM vs XGBoost vs AR(10)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'lstm_directional_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Individual model predictions vs actuals (time series) ---
fig, axes = plt.subplots(3, 1, figsize=(14, 14))

# Panel 1: LSTM vs Actual
axes[0].plot(test_dates, test_actuals, label='Actual', alpha=0.7, linewidth=1, color='#1f4e79')
axes[0].plot(test_dates, test_preds, label='LSTM', alpha=0.8, linewidth=1, color='#c55a11')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Log Return')
axes[0].set_title('LSTM Predictions vs Actual - Test Set')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Panel 2: XGBoost vs Actual
axes[1].plot(test_dates, xgb_test_actuals, label='Actual', alpha=0.7, linewidth=1, color='#1f4e79')
axes[1].plot(test_dates, xgb_test_preds, label='XGBoost', alpha=0.8, linewidth=1, color='#8e44ad')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Log Return')
axes[1].set_title('XGBoost Predictions vs Actual - Test Set')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Panel 3: AR(10) vs Actual
axes[2].plot(ar_test_dates, ar_y_test, label='Actual', alpha=0.7, linewidth=1, color='#1f4e79')
axes[2].plot(ar_test_dates, ar_preds, label='AR(10)', alpha=0.8, linewidth=1, color='#375623')
axes[2].set_xlabel('Date')
axes[2].set_ylabel('Log Return')
axes[2].set_title('AR(10) Predictions vs Actual - Test Set')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'individual_predictions_vs_actuals.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Residuals
lstm_resid = test_actuals - test_preds
xgb_resid = xgb_test_actuals - xgb_test_preds
ar_resid = ar_y_test - ar_preds

fig, axes = plt.subplots(3, 3, figsize=(16, 13))

# --- Row 1: LSTM residuals ---
axes[0, 0].plot(test_dates, lstm_resid, linewidth=0.7, color='#c55a11', alpha=0.8)
axes[0, 0].axhline(0, color='black', linewidth=0.5)
axes[0, 0].set_title('LSTM Residuals Over Time')
axes[0, 0].set_ylabel('Residual')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].hist(lstm_resid, bins=60, color='#c55a11', alpha=0.7, edgecolor='none', density=True)
axes[0, 1].set_title(f'LSTM Residual Distribution (std={lstm_resid.std():.4f})')
axes[0, 1].set_xlabel('Residual')
axes[0, 1].grid(True, alpha=0.3)

n = len(lstm_resid)
lstm_r = lstm_resid - lstm_resid.mean()
acf_full = np.correlate(lstm_r, lstm_r, mode='full')
acf_vals = acf_full[n-1:n+20] / acf_full[n-1]
ci = 1.96 / np.sqrt(n)
axes[0, 2].bar(range(21), acf_vals, color='#c55a11', alpha=0.7, width=0.6)
axes[0, 2].axhline(ci, color='red', linestyle='--', linewidth=0.8)
axes[0, 2].axhline(-ci, color='red', linestyle='--', linewidth=0.8)
axes[0, 2].set_title('LSTM Residual ACF')
axes[0, 2].set_xlabel('Lag')
axes[0, 2].grid(True, alpha=0.3)

# --- Row 2: XGBoost residuals ---
axes[1, 0].plot(test_dates, xgb_resid, linewidth=0.7, color='#8e44ad', alpha=0.8)
axes[1, 0].axhline(0, color='black', linewidth=0.5)
axes[1, 0].set_title('XGBoost Residuals Over Time')
axes[1, 0].set_ylabel('Residual')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].hist(xgb_resid, bins=60, color='#8e44ad', alpha=0.7, edgecolor='none', density=True)
axes[1, 1].set_title(f'XGBoost Residual Distribution (std={xgb_resid.std():.4f})')
axes[1, 1].set_xlabel('Residual')
axes[1, 1].grid(True, alpha=0.3)

m_xgb = len(xgb_resid)
xgb_r = xgb_resid - xgb_resid.mean()
acf_full_xgb = np.correlate(xgb_r, xgb_r, mode='full')
acf_vals_xgb = acf_full_xgb[m_xgb-1:m_xgb+20] / acf_full_xgb[m_xgb-1]
ci_xgb = 1.96 / np.sqrt(m_xgb)
axes[1, 2].bar(range(21), acf_vals_xgb, color='#8e44ad', alpha=0.7, width=0.6)
axes[1, 2].axhline(ci_xgb, color='red', linestyle='--', linewidth=0.8)
axes[1, 2].axhline(-ci_xgb, color='red', linestyle='--', linewidth=0.8)
axes[1, 2].set_title('XGBoost Residual ACF')
axes[1, 2].set_xlabel('Lag')
axes[1, 2].grid(True, alpha=0.3)

# --- Row 3: AR(10) residuals ---
axes[2, 0].plot(ar_test_dates, ar_resid, linewidth=0.7, color='#375623', alpha=0.8)
axes[2, 0].axhline(0, color='black', linewidth=0.5)
axes[2, 0].set_title('AR(10) Residuals Over Time')
axes[2, 0].set_ylabel('Residual')
axes[2, 0].grid(True, alpha=0.3)

axes[2, 1].hist(ar_resid, bins=60, color='#375623', alpha=0.7, edgecolor='none', density=True)
axes[2, 1].set_title(f'AR(10) Residual Distribution (std={ar_resid.std():.4f})')
axes[2, 1].set_xlabel('Residual')
axes[2, 1].grid(True, alpha=0.3)

m = len(ar_resid)
ar_r = ar_resid - ar_resid.mean()
acf_full_ar = np.correlate(ar_r, ar_r, mode='full')
acf_vals_ar = acf_full_ar[m-1:m+20] / acf_full_ar[m-1]
ci_ar = 1.96 / np.sqrt(m)
axes[2, 2].bar(range(21), acf_vals_ar, color='#375623', alpha=0.7, width=0.6)
axes[2, 2].axhline(ci_ar, color='red', linestyle='--', linewidth=0.8)
axes[2, 2].axhline(-ci_ar, color='red', linestyle='--', linewidth=0.8)
axes[2, 2].set_title('AR(10) Residual ACF')
axes[2, 2].set_xlabel('Lag')
axes[2, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

from scipy.stats import skew as sk, kurtosis as ku
print(f'{"":20s} {"Mean":>10} {"Std":>10} {"Skew":>10} {"Kurt":>10}')
print("-" * 62)
print(f'{"LSTM residuals":20s} {lstm_resid.mean():>10.6f} {lstm_resid.std():>10.6f} {sk(lstm_resid):>10.4f} {ku(lstm_resid):>10.4f}')
print(f'{"XGBoost residuals":20s} {xgb_resid.mean():>10.6f} {xgb_resid.std():>10.6f} {sk(xgb_resid):>10.4f} {ku(xgb_resid):>10.4f}')
print(f'{"AR(10) residuals":20s} {ar_resid.mean():>10.6f} {ar_resid.std():>10.6f} {sk(ar_resid):>10.4f} {ku(ar_resid):>10.4f}')

### Residual Analysis

## 14. Save Results

In [ ]:
# Save LSTM grid search results
results_df.to_csv(RESULTS_DIR / 'lstm_grid_search_results.csv', index=False)

# Save LSTM best hyperparameters
best_params = {
    'seq_len': best_sl,
    'hidden_size': best_hidden,
    'num_layers': best_layers,
    'dropout': best_dropout,
    'learning_rate': best_lr,
    'activation': best_act,
    'fc_hidden': best_fc,
    'batch_size': BATCH_SIZE,
    'epochs_trained': len(train_losses),
}
with open(RESULTS_DIR / 'lstm_best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)

# Save XGBoost grid search results
xgb_results_df.to_csv(RESULTS_DIR / 'xgb_grid_search_results.csv', index=False)

# Save XGBoost best hyperparameters
with open(RESULTS_DIR / 'xgb_best_params.json', 'w') as f:
    json.dump(best_xgb, f, indent=2)

# Save XGBoost model
final_xgb.save_model(str(RESULTS_DIR / 'xgb_model.json'))

# Save metrics
metrics_summary = pd.DataFrame([
    {'Model': 'LSTM', 'RMSE': test_metrics['RMSE'], 'MAE': test_metrics['MAE'], 'Dir_Acc': test_metrics['Dir_Acc']},
    {'Model': 'XGBoost', 'RMSE': xgb_rmse, 'MAE': xgb_mae, 'Dir_Acc': xgb_dir},
    {'Model': 'AR(10)', 'RMSE': ar_rmse, 'MAE': ar_mae, 'Dir_Acc': ar_dir},
    {'Model': 'Naive', 'RMSE': naive_rmse, 'MAE': naive_mae, 'Dir_Acc': naive_dir},
    {'Model': 'Persistence', 'RMSE': persist_rmse, 'MAE': persist_mae, 'Dir_Acc': persist_dir},
])
metrics_summary.to_csv(RESULTS_DIR / 'lstm_metrics.csv', index=False)

# Save LSTM model
torch.save(final_model.state_dict(), RESULTS_DIR / 'lstm_model.pth')

print('Saved:')
print(f'  LSTM grid search:    {RESULTS_DIR / "lstm_grid_search_results.csv"}')
print(f'  LSTM best params:    {RESULTS_DIR / "lstm_best_params.json"}')
print(f'  XGBoost grid search: {RESULTS_DIR / "xgb_grid_search_results.csv"}')
print(f'  XGBoost best params: {RESULTS_DIR / "xgb_best_params.json"}')
print(f'  XGBoost model:       {RESULTS_DIR / "xgb_model.json"}')
print(f'  Metrics summary:     {RESULTS_DIR / "lstm_metrics.csv"}')
print(f'  LSTM model weights:  {RESULTS_DIR / "lstm_model.pth"}')

## 15. Summary

| Metric | LSTM | XGBoost | AR(10) | Naive (predict 0) | Persistence |
|--------|------|---------|--------|-------------------|-------------|
| RMSE   | -    | -       | -      | -                 | -           |
| MAE    | -    | -       | -      | -                 | -           |
| Dir Acc| -    | -       | -      | -                 | -           |

*(Values filled after running the notebook)*